# 0027 / 05 Dual-T4 fine-tuning

Load 0025 V4 best-greedy, verify its SHA-256, and fine-tune the canonical policy on two Kaggle T4 GPUs.


In [ ]:
from __future__ import annotations
import gzip, hashlib, importlib, json, sys
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, IterableDataset, get_worker_info

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("0027 requires Kaggle NvidiaTeslaT4 x2")
print({"cuda_count": torch.cuda.device_count(), "devices": [torch.cuda.get_device_name(i) for i in range(2)]})

input_root = Path("/kaggle/input")
manifest_candidates = []
for path in input_root.rglob("manifest.json"):
    try:
        candidate = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if (candidate.get("schema_version") == "0025_canonical_semantic_decision_v2"
            and candidate.get("status") == "complete"
            and "initialized_from_0025" in candidate
            and set(candidate.get("shards", {})) >= {"train", "validation"}):
        manifest_candidates.append((path, candidate))
if len(manifest_candidates) != 1:
    raise FileNotFoundError(f"expected exactly one notebook 04 dataset manifest, found {len(manifest_candidates)}")
dataset_manifest, manifest = manifest_candidates[0]
dataset_root = dataset_manifest.parent
source_candidates = sorted({
    path.parent.parent
    for path in input_root.rglob("official_public_prototypes_v1.json")
    if path.parent.name == "assets" and (path.parent.parent / "model" / "canonical").is_dir()
})
if len(source_candidates) != 1:
    raise FileNotFoundError("attach the vendored 0025 source package")
source_root = source_candidates[0]
sys.path.insert(0, str(source_root.parent))
package = source_root.name
PrototypeIndex = importlib.import_module(f"{package}.features.prototypes").PrototypeIndex
collate_canonical_records = importlib.import_module(f"{package}.features.canonical.batching").collate_canonical_records
canonical_model = importlib.import_module(f"{package}.model.canonical")
CanonicalModelConfig, CanonicalSemanticPolicy = canonical_model.CanonicalModelConfig, canonical_model.CanonicalSemanticPolicy

class Rows(IterableDataset):
    def __init__(self, paths):
        self.paths = tuple(paths)

    def __iter__(self):
        worker = get_worker_info()
        paths = self.paths if worker is None else self.paths[worker.id::worker.num_workers]
        for path in paths:
            with gzip.open(path, "rt", encoding="utf-8") as handle:
                for line in handle:
                    if line.strip():
                        yield json.loads(line)

train_paths = [dataset_root / item["path"] for item in manifest["shards"]["train"]]
valid_paths = [dataset_root / item["path"] for item in manifest["shards"]["validation"]]
prototype_path = source_root / "assets" / "official_public_prototypes_v1.json"
prototypes = PrototypeIndex.load(prototype_path)
policy = CanonicalSemanticPolicy(CanonicalModelConfig(), prototypes).to("cuda:0")

expected_sha = "adc4eaeca1e62a28bbd762e8e513212c94941044bbc85673f02bbeadc1865aa3"
expected_bytes = 129028675
checkpoints = [path for path in input_root.rglob("best_greedy_exact.pt") if path.is_file() and path.stat().st_size == expected_bytes]
if not checkpoints:
    raise FileNotFoundError("attach 0025 V4 best_greedy_exact.pt (129028675 bytes)")
checkpoint = checkpoints[0]
actual_sha = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
if actual_sha != expected_sha:
    raise ValueError(f"checkpoint sha mismatch: {actual_sha}")
payload = torch.load(checkpoint, map_location="cpu")
policy.load_state_dict(payload["state_dict"], strict=True)

class TeacherLogits(nn.Module):
    def __init__(self, policy):
        super().__init__()
        self.policy = policy

    def forward(self, batch):
        return self.policy.teacher_logits(batch)

distributed = nn.DataParallel(TeacherLogits(policy), device_ids=[0, 1])
optimizer = torch.optim.AdamW(distributed.parameters(), lr=1e-4, weight_decay=0.02)
scaler = torch.amp.GradScaler("cuda", enabled=True)
amp_dtype = torch.float16  # Nvidia T4 Tensor Cores support FP16, not BF16.

def loader(paths, batch_size):
    return DataLoader(Rows(paths), batch_size=batch_size, num_workers=2, pin_memory=True, collate_fn=collate_canonical_records)

train_loader, valid_loader = loader(train_paths, 192), loader(valid_paths, 256)
def move(batch):
    return {key: value.to("cuda:0", non_blocking=True) for key, value in batch.items()}

out = Path("/kaggle/working/ptcg_0027_dual_t4_train")
out.mkdir(parents=True, exist_ok=True)
history = []
for epoch in range(1, 11):
    distributed.train()
    train_loss_sum, train_tokens = 0.0, 0
    for raw in train_loader:
        batch = move(raw)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits = distributed(batch)
            mask = batch["targets"].ne(-100)
            loss = F.cross_entropy(logits[mask], batch["targets"][mask])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(distributed.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        token_count = int(mask.sum().item())
        train_loss_sum += float(loss.detach().cpu()) * token_count
        train_tokens += token_count
    distributed.eval()
    valid_loss_sum, valid_tokens = 0.0, 0
    with torch.inference_mode():
        for raw in valid_loader:
            batch = move(raw)
            logits = distributed(batch)
            mask = batch["targets"].ne(-100)
            token_count = int(mask.sum().item())
            valid_loss_sum += float(F.cross_entropy(logits[mask], batch["targets"][mask]).cpu()) * token_count
            valid_tokens += token_count
    row = {"epoch": epoch, "train_loss": train_loss_sum / max(train_tokens, 1), "validation_loss": valid_loss_sum / max(valid_tokens, 1), "gpu_count": 2, "initialized_from_sha256": expected_sha}
    history.append(row)
    print(row, flush=True)
    torch.save({"schema_version": "0027_model_only_checkpoint_v1", "state_dict": distributed.module.policy.state_dict(), "metadata": row}, out / "last_model.pt")
(out / "training_report.json").write_text(json.dumps({"experiment": "0027_semantic_foundation_pretraining", "initialized_from": "0025/V4/best_greedy_exact.pt", "checkpoint_sha256": expected_sha, "gpu_count": 2, "history": history}, indent=2) + "\n")

